# Customer Churn Prediction — Model Inference

This notebook is the inference stage of the customer churn project.

The ANN has already been trained in the previous notebook. Here, I will load that trained model along with the preprocessing objects and use them to predict churn for a new customer profile.

The important part of inference is consistency. The new customer data must go through the same encoding, feature construction, ordering, and scaling steps that were used during training.

### In this notebook

- Load the trained ANN
- Load the saved encoders and scaler
- Create a sample customer profile
- Encode categorical features
- Reconstruct the model input
- Apply the saved scaler
- Generate a churn probability
- Convert the probability into a simple business interpretation

This notebook is essentially the bridge between the trained model and the Streamlit application that uses the same prediction workflow interactively.

In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

2026-09-14 04:58:04.240915: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-14 04:58:08.228177: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-14 04:58:11.057336: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-14 04:58:14.911651: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-14 04:58:14.912242: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-14 04:58:20.179109: I tensorflow/core/platform/cpu_feature_guard.cc:

## 1. Load the Trained Model and Preprocessing Objects

The model itself is not enough to make reliable predictions.

I also need the preprocessing objects created during training — the Gender encoder, Geography encoder, and StandardScaler — so that a new customer is transformed in exactly the same way as the training data.

In [2]:
# load the trained model, scaler, pickle, OneHot encoder
model = load_model('model.h5', compile=False)

# load the encoders and scaler

with open('one_hot_encoder_geography.pkl', 'rb') as file:
    one_hot_encoder_geography = pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

## 2. Create a New Customer Profile

To test the trained model, I am creating a sample customer profile containing the same business features used during training.

At this stage, the values are still in their human-readable form — for example, `France` for Geography and `Male` for Gender.

In [3]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

## 3. Apply the Same Preprocessing Used During Training

The model cannot work directly with the raw categorical values.

I will first encode the categorical features and then combine them with the remaining numerical features. After that, the complete feature set will be arranged in the expected order and scaled using the scaler fitted during training.

In [4]:
# One-hot encode 'Geography'
geo_encoded = one_hot_encoder_geography.transform(
    [[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(
    geo_encoded, columns=one_hot_encoder_geography.get_feature_names_out(['Geography']))
geo_encoded_df

/workspaces/Ann-churn-prediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [5]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [6]:
# Encode categorical variables
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [7]:
# concatenation one hot encoded
input_df = pd.concat(
    [input_df.drop("Geography", axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [8]:
# Scaling the input data
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

## 4. Generate the Churn Prediction

The customer profile has now been transformed into the same numerical representation expected by the ANN.

I can now pass the processed input to the trained model and obtain the predicted probability of churn.

In [9]:
# PRedict churn
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step


array([[0.02332283]], dtype=float32)

### Churn Probability

The model returns a probability between 0 and 1.

I will extract that value so that it can be interpreted more easily and later displayed in the Streamlit application.

In [10]:
prediction_proba = prediction[0][0]

In [11]:
prediction_proba

0.023322828

## 5. Interpreting the Prediction

For this demonstration, I am using a threshold of `0.5` to convert the predicted probability into a simple churn / no-churn interpretation.

This threshold is only a demonstration choice. It should not be treated as a universal business decision threshold without validating the model and business requirements.

In [12]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.


## 6. From Notebook to Application

The inference process demonstrated here is the same foundation used by the Streamlit application.

The application takes customer information from the user, applies the saved preprocessing steps, sends the processed features to the ANN, and presents the resulting churn risk in a more accessible interface.

This completes the transition from model experimentation to an interactive ML application.